# 03 — Train and compare (protocol c4-env-eval-v1.1)

Uses simulator artifacts from notebooks 01–02. **Reuse notebook 01 artifacts** after a code-only ZIP reload; do not refit the response model unless artifacts are missing.

Default comparison:
- `dqn` / `ppo` — vanilla SB3; **no** mask; penalties only
- `weakest_mastery` — lowest current mastery (masked)
- `random_valid` — seeded random valid (masked)
- `recent_performance` — lowest rolling in-episode accuracy (masked); cold-start → untried

Opt-in: add `maskable_ppo` for sb3-contrib MaskablePPO (masks in train **and** eval).

New runs write `train_*_c4-env-eval-v1.1_*` folders and **do not overwrite** historical metrics.
Scenario names only change `prior_strength` presets (not held-out real learners).

- `RUN_MODE = "smoke"` — short CPU run (safe default).
- `RUN_MODE = "full"` — longer training, multiple seeds. **Set manually; not auto-started.**

Optional: `training.diagnostics.run_checkpoint_diagnostics` for validation-only traces on old checkpoints.


In [ ]:
# === USER CONFIG ===
RUN_IN_COLAB = True
DRIVE_SUBDIR = "AdaptiveLearnSE/c4_runs"
RUN_MODE = "smoke"  # change to "full" only when ready
DEVICE = "auto"  # auto -> cpu for small MLP policies; set "cuda" if desired
C4_ROOT = None
# Defaults: dqn, ppo, weakest_mastery, random_valid, recent_performance
# Opt-in mask-aware: TrainConfig(..., algorithms=(..., "maskable_ppo"))


In [ ]:
from pathlib import Path
import sys
import subprocess

if C4_ROOT is None:
    C4_ROOT = Path("..").resolve()
sys.path.insert(0, str(C4_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(C4_ROOT / "requirements-training.txt")])

from training.paths import resolve_output_root, stage_dirs
from training.colab_bootstrap import mount_drive
from training.config import EnvConfig, TrainConfig
from training.baselines import COMPARISON_LABELS, COMPARISON_NOTES
from training.pipeline import load_simulator_from_artifacts, require_previous_stage
from training.train_rl import run_training_comparison, resume_instructions

if RUN_IN_COLAB:
    try:
        mount_drive()
    except Exception as e:
        print(e)

OUT = resolve_output_root(DRIVE_SUBDIR if RUN_IN_COLAB else None, prefer_drive=RUN_IN_COLAB)
DIRS = stage_dirs(OUT)
ARTIFACTS = require_previous_stage(DIRS["artifacts"] / "response_model.joblib", "01", "Run notebook 01 once; reuse artifacts after code ZIP reload.").parent
require_previous_stage(DIRS["synthetic"] / "simulator_assumptions.json", "02", "Run notebook 02 (or reuse synthetic/).")

env_cfg = EnvConfig()
sim = load_simulator_from_artifacts(ARTIFACTS, env_cfg)
train_cfg = TrainConfig(mode=RUN_MODE, device=DEVICE)
resolved = train_cfg.resolved()
print("Resolved train config:", resolved)
print("Algorithms:", resolved.algorithms)
print("Labels:", {k: COMPARISON_LABELS[k] for k in resolved.algorithms})
print("Notes:", COMPARISON_NOTES)
print("Resume help:\n", resume_instructions(DIRS["checkpoints"]))


In [ ]:
# Smoke by default. Full mode only if RUN_MODE="full".
# Writes a NEW train_* folder; does not overwrite previous smoke outputs.
results = run_training_comparison(sim, env_cfg, train_cfg, DIRS["root"])
print("Run dir:", results["run_dir"])
print("Validation selection:", results["selection"])
print("Do not interpret these as SLIIT SE gains. Numbers come from the simulator.")
print("Do not claim improvement vs prior smoke until you compare the new metrics explicitly.")
